# Causal Inference in Practice
## Week 14 — Advanced Topics · Practice Notebook

> **Block IV — Modern methods & application**
>
> Mediation, sensitivity analysis, time-varying treatments, and discovering structure from data.

**How to use this notebook.** Run the cells top to bottom. Sections marked
**🔧 Exercise** contain a `# TODO` for you to complete; a matching
**✅ Solution** cell follows (collapsed in spirit — try it yourself first).
Every dataset here is *simulated with a known ground truth*, so you can always
check whether your estimate recovered the right answer.

*Estimated time: 60–90 minutes. Toolkit: `numpy`, `pandas`, `statsmodels`,
`scikit-learn`, `matplotlib` — all standard.*

---


In [ ]:
# --- Environment check & shared setup -------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 11})

RNG = np.random.default_rng(7)   # one seed for the whole notebook → reproducible
print("Environment OK — numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Mediation — natural direct & indirect effects

We simulate the canonical mediation structure: a treatment `X` affects a mediator `M`, and both `X` and `M` affect the outcome `Y`. With **known** coefficients we can check that our estimates recover the true direct and indirect effects — and that they sum to the total effect.

Truth:  `X → M` has slope `a`, `M → Y` has slope `b`, and the direct path `X → Y` has slope `c'`. Then

- **NIE** (natural indirect effect) `= a · b`  — the route `X → M → Y`,
- **NDE** (natural direct effect) `= c'`  — the route that skips `M`,
- **Total effect** `= NDE + NIE = c' + a·b`.

In [ ]:
import statsmodels.api as sm

# --- ground truth ---
a, b, cprime = 0.6, 0.5, 0.3        # X->M, M->Y, direct X->Y
n = 40_000
X = RNG.normal(size=n)
M = a * X + RNG.normal(size=n)
Y = cprime * X + b * M + RNG.normal(size=n)

# --- product / difference method with linear models ---
a_hat = sm.OLS(M, sm.add_constant(X)).fit().params[1]      # X -> M
outcome = sm.OLS(Y, sm.add_constant(np.c_[X, M])).fit().params
NDE, b_hat = outcome[1], outcome[2]                        # direct, M -> Y
NIE = a_hat * b_hat                                        # indirect
TE_direct = sm.OLS(Y, sm.add_constant(X)).fit().params[1]  # total, X alone

print(f'a (X->M) : est={a_hat:.3f}  true={a}')
print(f'b (M->Y) : est={b_hat:.3f}  true={b}')
print(f'NDE      : est={NDE:.3f}   true={cprime}')
print(f'NIE      : est={NIE:.3f}   true={a*b:.3f}')
print(f'NDE+NIE  : {NDE + NIE:.3f}   total (Y~X): {TE_direct:.3f}'
      f'   true total: {cprime + a*b:.3f}')

The decomposition holds: `NDE + NIE` equals the total effect, and each piece recovers its true coefficient. Let's make that a hard check.

In [ ]:
assert abs(NDE - cprime) < 0.05,         'NDE should recover c'
assert abs(NIE - a * b) < 0.05,          'NIE should recover a*b'
assert abs((NDE + NIE) - TE_direct) < 0.05, 'NDE+NIE should equal TE'
print('OK — mediation decomposition recovered within tolerance.')
print(f'Proportion mediated = NIE/TE = {NIE / (NDE + NIE):.1%}')

### 🔧 Exercise 1.1 — a mediator that does nothing

Suppose `M` is influenced by `X` but does **not** affect `Y` (set `b = 0`). Predict the NIE *before* you run it, then estimate NDE and NIE and confirm the indirect effect is ~0 while the direct effect equals the (now total) `X → Y` slope.

Fill in the `# TODO`s. The skeleton still runs as-is.

In [ ]:
# TODO: simulate X -> M, but M -> Y has slope 0; keep a direct X -> Y = 0.4
a2, b2, cprime2 = 0.6, 0.0, 0.4
X2 = RNG.normal(size=n)
M2 = ...        # TODO: a2 * X2 + noise
Y2 = ...        # TODO: cprime2 * X2 + b2 * M2 + noise
# a2_hat = ...  # slope of M2 on X2
# out2   = ...  # OLS of Y2 on [X2, M2]; params
# NDE2, NIE2 = ..., ...
# print(NDE2, NIE2)

### ✅ Solution 1.1

In [ ]:
a2, b2, cprime2 = 0.6, 0.0, 0.4
X2 = RNG.normal(size=n)
M2 = a2 * X2 + RNG.normal(size=n)
Y2 = cprime2 * X2 + b2 * M2 + RNG.normal(size=n)
a2_hat = sm.OLS(M2, sm.add_constant(X2)).fit().params[1]
out2   = sm.OLS(Y2, sm.add_constant(np.c_[X2, M2])).fit().params
NDE2, NIE2 = out2[1], a2_hat * out2[2]
print(f'NDE2={NDE2:.3f} (true {cprime2})   NIE2={NIE2:.3f} (true 0.0)')
assert abs(NIE2) < 0.05,            'no M->Y means NIE ~ 0'
assert abs(NDE2 - cprime2) < 0.05,  'direct effect = total here'
print('OK — a mediator off the outcome path carries no indirect effect.')

## 2 · Sensitivity analysis — the E-value

An observational estimate always assumes *no unmeasured confounding*. The **E-value** quantifies how strong an unmeasured confounder would have to be — associated with **both** treatment and outcome — to fully explain away an observed risk ratio.

For an observed risk ratio `RR ≥ 1`:

$$E = RR + \sqrt{RR\,(RR - 1)}$$

(if `RR < 1`, apply the formula to `1/RR`). A bigger E-value means a more robust finding.

In [ ]:
def evalue_rr(rr):
    """E-value for an observed risk ratio (point estimate)."""
    rr = rr if rr >= 1 else 1.0 / rr
    return rr + np.sqrt(rr * (rr - 1.0))

for rr in (1.0, 1.2, 2.0, 3.9):
    print(f'RR = {rr:>3}  ->  E-value = {evalue_rr(rr):.2f}')

# sanity: RR=1 (no effect) gives E=1; RR=3.9 is VanderWeele's classic ~7.26
assert abs(evalue_rr(1.0) - 1.0) < 1e-9
assert abs(evalue_rr(3.9) - 7.26) < 0.01
print('OK — E-value formula matches known values.')

Now connect the E-value to a **simulated confounded estimate**. We build a binary world with a confounder `U` of known strength, compute the *crude* (confounded) risk ratio, and show its E-value tells us how strong a hidden `U` would need to be to produce it from nothing.

In [ ]:
# A confounded world: U raises both treatment T and outcome D.
# The TRUE causal effect of T on D is zero (D does not depend on T).
n = 50_000
U  = RNG.binomial(1, 0.5, n)                       # hidden confounder
T  = RNG.binomial(1, 0.2 + 0.6 * U)                # U -> T
D  = RNG.binomial(1, 0.1 + 0.5 * U)                # U -> D, NOT T -> D

p1 = D[T == 1].mean()      # crude risk in treated
p0 = D[T == 0].mean()      # crude risk in untreated
crude_rr = p1 / p0
print(f'crude (confounded) RR = {crude_rr:.2f}  '
      f'(true causal RR = 1.00)')
print(f'E-value of the crude RR = {evalue_rr(crude_rr):.2f}')
print('Reading: a confounder this strong on BOTH arms is exactly what U is.')

### 🔧 Exercise 2.1 — what E-value would you need?

A colleague reports an adjusted risk ratio of **RR = 1.5** and calls it 'solid evidence.' Compute its E-value. Then decide: if every measured confounder in the field has associations no stronger than ~1.4 with treatment and outcome, is this result robust?

Fill in the `# TODO`.

In [ ]:
# TODO: compute the E-value for RR = 1.5 and compare to 1.4
rr_obs = 1.5
ev = ...                 # TODO: use evalue_rr
# robust = ...           # TODO: is ev comfortably above 1.4?
# print(ev, robust)

### ✅ Solution 2.1

In [ ]:
rr_obs = 1.5
ev = evalue_rr(rr_obs)
robust = ev > 1.4
print(f'E-value for RR={rr_obs}: {ev:.2f}')
print(f'Above the ~1.4 ceiling of measured confounders? {robust}')
assert abs(ev - (1.5 + np.sqrt(1.5 * 0.5))) < 1e-9
print('A confounder of strength ~2.37 on both arms would be needed — '
      'stronger than anything measured, so the finding is fairly robust\n'
      'but NOT immune: the margin over 1.4 is not huge.')

## 3 · g-formula — time-varying confounding

The hardest idea of the week. Treatment is given at two times, `A0` then `A1`, with a covariate `L1` measured in between. The trap:

- `A0 → L1`  (treatment moves the biomarker),
- `L1 → A1`  (the clinician reacts to it),
- `L1 → Y` and `A0, A1 → Y`.

So `L1` is a **confounder of `A1`** and a **mediator of `A0`** at once. We compare the always-treat plan `(A0=A1=1)` to never-treat `(A0=A1=0)`; with these coefficients the **true** contrast is `1 + 1 + 1·(1−0) = 3`.

In [ ]:
n = 200_000
A0 = RNG.binomial(1, 0.5, n)
L1 = RNG.normal(1.0 * A0, 1.0)                     # A0 -> L1
A1 = RNG.binomial(1, 1.0 / (1.0 + np.exp(-(L1 - 0.5))))  # L1 -> A1
Y  = 1.0 * A0 + 1.0 * A1 + 1.0 * L1 + RNG.normal(size=n) # truth: 3
TRUE = 3.0

# --- ordinary regressions: both wrong ---
adj = sm.OLS(Y, sm.add_constant(np.c_[A0, A1, L1])).fit().params
omit = sm.OLS(Y, sm.add_constant(np.c_[A0, A1])).fit().params
print(f'TRUE effect (always vs never)      : {TRUE:.2f}')
print(f'regression ADJUSTING for L1 (A0+A1): {adj[1] + adj[2]:.2f}  '
      f'(biased DOWN: blocks A0->L1->Y)')
print(f'regression OMITTING  L1 (A0+A1)    : {omit[1] + omit[2]:.2f}  '
      f'(biased UP: A1 left confounded)')

Neither regression is right — one over-controls `A0`, the other leaves `A1` confounded. The **g-formula** fixes this by *standardization*: model how `L1` responds to `A0`, model `Y` from the full history, then **simulate** each treatment plan and average.

In [ ]:
# g-formula by standardization (g-computation)
pL = sm.OLS(L1, sm.add_constant(A0)).fit().params           # E[L1 | A0]
pY = sm.OLS(Y,  sm.add_constant(np.c_[A0, A1, L1])).fit().params

def g_mean(a0, a1):
    L1_sim = pL[0] + pL[1] * a0          # simulate L1 under do(A0=a0)
    return pY[0] + pY[1]*a0 + pY[2]*a1 + pY[3]*L1_sim

g_effect = g_mean(1, 1) - g_mean(0, 0)
print(f'g-formula (always vs never): {g_effect:.2f}   true: {TRUE:.2f}')
assert abs(g_effect - TRUE) < 0.1,        'g-formula should recover ~3'
assert abs((adj[1] + adj[2]) - TRUE) > 0.5, 'adjusting for L1 is biased'
print('OK — the g-formula recovers the truth where regression cannot.')

### 🔧 Exercise 3.1 — turn off the feedback

If `A0` did **not** affect `L1` (no `A0 → L1` arrow), then `L1` is an ordinary confounder and plain regression adjusting for `L1` should be fine. Re-simulate with `L1` independent of `A0` and confirm the adjusted regression now matches the g-formula.

Fill in the `# TODO`s.

In [ ]:
# TODO: same world but break A0 -> L1 (make L1 not depend on A0)
A0b = RNG.binomial(1, 0.5, n)
L1b = ...        # TODO: RNG.normal(0.0, 1.0, n)  — no dependence on A0b
# TODO: uncomment once L1b is real:
# A1b = RNG.binomial(1, 1.0 / (1.0 + np.exp(-(L1b - 0.5))))
# Yb  = 1.0 * A0b + 1.0 * A1b + 1.0 * L1b + RNG.normal(size=n)
# now the true effect is just 1 + 1 = 2 (L1 no longer carries A0's effect)
# adjb = ...     # OLS of Yb on [A0b, A1b, L1b]; params
# print(adjb[1] + adjb[2])

### ✅ Solution 3.1

In [ ]:
A0b = RNG.binomial(1, 0.5, n)
L1b = RNG.normal(0.0, 1.0, n)                      # NO A0 -> L1 arrow
A1b = RNG.binomial(1, 1.0 / (1.0 + np.exp(-(L1b - 0.5))))
Yb  = 1.0 * A0b + 1.0 * A1b + 1.0 * L1b + RNG.normal(size=n)
TRUE_b = 2.0                                       # 1 (A0) + 1 (A1)
adjb = sm.OLS(Yb, sm.add_constant(np.c_[A0b, A1b, L1b])).fit().params
print(f'adjusting for L1 now: {adjb[1] + adjb[2]:.2f}   true: {TRUE_b}')
assert abs((adjb[1] + adjb[2]) - TRUE_b) < 0.1
print('OK — with no treatment->confounder feedback, regression is fine.')
print('That feedback arrow is the whole reason g-methods exist.')

## 4 · Causal discovery — a PC-style skeleton test

Causal discovery learns graph structure from data. The first move of the **PC algorithm** is to delete an edge `X–Y` whenever `X` and `Y` are independent given some conditioning set. For continuous Gaussian data, a **partial correlation** is the conditional-independence test.

We generate the chain `X → Y → Z`. The truth: `X` and `Z` are correlated *marginally* but **independent given `Y`** — so PC should remove the `X–Z` edge and keep the chain skeleton `X – Y – Z`.

In [ ]:
n = 4000
Xc = RNG.normal(size=n)
Yc = Xc + RNG.normal(size=n)         # X -> Y
Zc = Yc + RNG.normal(size=n)         # Y -> Z   (chain)

def pcorr(u, v, given=None):
    """(Partial) correlation of u and v, optionally given `given`."""
    if given is None:
        return np.corrcoef(u, v)[0, 1]
    W = sm.add_constant(given)
    ru = u - W @ np.linalg.lstsq(W, u, rcond=None)[0]
    rv = v - W @ np.linalg.lstsq(W, v, rcond=None)[0]
    return np.corrcoef(ru, rv)[0, 1]

print(f'corr(X, Z)        = {pcorr(Xc, Zc):+.3f}   (marginally linked)')
print(f'corr(X, Z | Y)    = {pcorr(Xc, Zc, Yc):+.3f}   (vanishes -> drop edge)')
print(f'corr(X, Y | Z)    = {pcorr(Xc, Yc, Zc):+.3f}   (stays -> keep edge)')
print(f'corr(Y, Z | X)    = {pcorr(Yc, Zc, Xc):+.3f}   (stays -> keep edge)')

In [ ]:
# Build the skeleton: keep an edge iff some test does NOT vanish.
thr = 0.05
edges = {}
edges['X-Z'] = abs(pcorr(Xc, Zc, Yc)) > thr   # conditioned on Y
edges['X-Y'] = abs(pcorr(Xc, Yc, Zc)) > thr   # conditioned on Z
edges['Y-Z'] = abs(pcorr(Yc, Zc, Xc)) > thr   # conditioned on X
kept = [e for e, keep in edges.items() if keep]
print('skeleton edges kept:', kept)
assert edges['X-Y'] and edges['Y-Z'] and not edges['X-Z'], \
    'PC should recover the chain skeleton X-Y-Z'
print('OK — recovered the chain skeleton; the X-Z edge was correctly removed.')

**The limit, in one line.** The chain `X→Y→Z`, the reversed chain `X←Y←Z`, and the fork `X←Y→Z` all imply the *same* independence `X ⫫ Z | Y`. Observational data alone cannot tell them apart — they form one **Markov equivalence class**. PC returns the skeleton (and any colliders it can orient), not a unique DAG. Orienting the rest needs interventions, time order, or extra assumptions (the territory of GES and NOTEARS).

## 5 · Wrap-up & self-check

- **Mediation** splits a total effect into NDE + NIE; the product method gives `NIE = a·b`, and the pieces sum to the total — *if* the mediator–outcome relationship is unconfounded.
- **E-values** turn 'what about unmeasured confounding?' into a number: `E = RR + √(RR(RR−1))`. Bigger = more robust.
- **g-formula**: when a confounder is affected by prior treatment, no single regression is right; standardize by simulating the intervention. You saw both regressions fail and the g-formula recover the truth.
- **Causal discovery** (PC) recovers a skeleton and a Markov equivalence class — not a unique DAG — from independence tests alone.

**You're ready for Week 15** if you can decompose a total effect, compute and read an E-value, explain why time-varying confounding defeats regression, and say what an equivalence class is. Next week: the capstone — assembling the full workflow, reproducibly, and communicating a causal claim honestly.